# HC-MARL Scaling Experiments (N={3,4,6,8,12} x 5 seeds)

**File:** #188 from HC-MARL battle plan

Test HC-MARL performance across different worker counts N={3,4,6,8,12} x 5 seeds.

**Requirements:** Colab Pro with T4 GPU (or Kaggle T4 P100)

**Estimated time:** 3-4 hours per seed on T4

## Step 1: Install

In [ ]:
# Install dependencies
# S5: use Colab-default torch (matches runtime CUDA); manual cu121 pin causes drift / CPU fallback
!pip install torchvision 2>/dev/null || true
!pip install gymnasium pettingzoo scipy cvxpy numpy pandas matplotlib pyyaml wandb nbformat jupyter
!pip install omnisafe safety-gymnasium 2>/dev/null || echo "OmniSafe/Safety-Gymnasium optional"


## Step 2: Clone

In [ ]:
# Clone repository (change URL to your repo)
import os

REPO_URL = "https://github.com/ADITYA-WORK-MAITI/hcmarl-project.git"  # <-- UPDATE THIS
PROJECT_DIR = "/content/hcmarl_project"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

os.chdir(PROJECT_DIR)
!pip install -e . 2>/dev/null || echo "Package install skipped"

# Verify
!python -c "import hcmarl; print('HC-MARL package OK')"
!python -c "import torch; print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')"

## Step 3: Drive

In [ ]:
# Mount Google Drive for persistent checkpoint storage
from google.colab import drive
drive.mount('/content/drive')

# Create output directory on Drive
DRIVE_DIR = "/content/drive/MyDrive/hcmarl_results"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Results will be saved to: {DRIVE_DIR}")
# Assert Drive is actually writable (Colab sometimes fails to mount silently)
assert os.path.isdir('/content/drive/MyDrive'), 'Drive mount failed — re-run this cell'
BACKUP_DIR = '/content/drive/MyDrive/hcmarl_backup'
os.makedirs(BACKUP_DIR, exist_ok=True)
print(f'Checkpoints will mirror to: {BACKUP_DIR}')


## GPU Check

In [ ]:
# Verify GPU is available
import torch
assert torch.cuda.is_available(), 'No GPU detected! Enable GPU in Runtime > Change runtime type'
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'GPU: {gpu} ({mem:.1f} GB)')


## Training

In [ ]:
# S9: re-assert Drive is mounted right before training — Colab can silently
# unmount between the Drive cell and here, which would make --drive-backup-dir
# create a local dir that evaporates on disconnect.
import os, sys
if not os.path.isdir("/content/drive/MyDrive"):
    raise RuntimeError("Google Drive is not mounted — re-run the Drive mount cell before training.")

# Run all training jobs
!python scripts/run_scaling.py --device cuda --drive-backup-dir /content/drive/MyDrive/hcmarl_backup


### Or run individual seeds

In [ ]:
# Run a single scaling config
# !python scripts/train.py --config config/scaling_n3.yaml --method hcmarl --seed 0 --device cuda
# !python scripts/train.py --config config/scaling_n6.yaml --method hcmarl --seed 0 --device cuda
# !python scripts/train.py --config config/scaling_n12.yaml --method hcmarl --seed 0 --device cuda


## Evaluation

In [ ]:
# Evaluate trained models
import glob, json

checkpoints = glob.glob('checkpoints/*/seed_*/checkpoint_final.pt')
print(f'Found {len(checkpoints)} checkpoints')

for ckpt in sorted(checkpoints):
    parts = ckpt.replace('\\', '/').split('/')
    method = parts[1]
    seed = parts[2].split('_')[1]
    # Use the saved training config (has MMICRL results + correct env settings)
    log_dir = ckpt.replace('checkpoints', 'logs').replace('checkpoint_final.pt', '')
    saved_cfg = log_dir + 'config.yaml'
    import os
    if os.path.exists(saved_cfg):
        config_flag = saved_cfg
    else:
        config_flag = f'config/{method}_config.yaml' if method != 'hcmarl' else 'config/hcmarl_full_config.yaml'
    print(f'\nEvaluating {method} seed={seed} with {config_flag}...')
    !python scripts/evaluate.py --checkpoint {ckpt} --config {config_flag} --method {method} --seed {seed} --n-episodes 100

## Save Results to Drive

In [ ]:
# Copy all results to Google Drive
import shutil
for folder in ['checkpoints', 'logs', 'results']:
    src = os.path.join(PROJECT_DIR, folder)
    dst = os.path.join(DRIVE_DIR, folder)
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Copied {folder}/ to Drive')
print('\nAll results saved to Google Drive!')
